In [1]:
import os
import wfdb
import pandas as pd

In [2]:
input_path = '../data/'
output_path = '../mitbih_database/'

os.makedirs(output_path, exist_ok=True)

In [3]:
record_names = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(input_path)
    if f.endswith('.hea')
])

In [4]:
print(f'Liczba rekordów: {len(record_names)}')
print(f'Rekordy: {record_names}')

Liczba rekordów: 48
Rekordy: ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']


In [ ]:
aami_mapping = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N', # uderzenia przypisywane do N
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S', # uderzenia przypisywane do S
    'V': 'V', 'E': 'V', # uderzenia przypisywane do V
    'F': 'F', # uderzenia przypisywane do F
    '/': 'Q', 'f': 'Q', 'Q': 'Q' # uderzenia przypisywane do Q
}

In [ ]:
for record_name in record_names:
    record_path = os.path.join(input_path, record_name)
    
    # zapisanie wartości sygnału do csv
    record = wfdb.rdrecord(record_path)
    
    ecg = pd.DataFrame(
        record.p_signal,
        columns=record.sig_name
    )
    ecg.insert(0, 'sample #', range(len(ecg)))
    
    ecg_path = os.path.join(output_path, f'{record_name}_record.csv')
    ecg.to_csv(ecg_path, index=False)
    
    # zapisanie adnotacji do drugiego csv
    annotation = wfdb.rdann(record_path, 'atr')
    
    annotations = pd.DataFrame({
        'Sample #': annotation.sample,
        'Type': annotation.symbol
    })
    
    annotations['AAMI'] = annotations['Type'].map(aami_mapping)
    annotations.insert(0, 'Record', record_name)
    
    ann_path = os.path.join(output_path, f'{record_name}_annotation.csv')
    annotations.to_csv(ann_path, index=False)
    
    print(f'zakończono przetwarzanie rekordu {record_name}')

100 -> done
101 -> done
102 -> done
103 -> done
104 -> done
105 -> done
106 -> done
107 -> done
108 -> done
109 -> done
111 -> done
112 -> done
113 -> done
114 -> done
115 -> done
116 -> done
117 -> done
118 -> done
119 -> done
121 -> done
122 -> done
123 -> done
124 -> done
200 -> done
201 -> done
202 -> done
203 -> done
205 -> done
207 -> done
208 -> done
209 -> done
210 -> done
212 -> done
213 -> done
214 -> done
215 -> done
217 -> done
219 -> done
220 -> done
221 -> done
222 -> done
223 -> done
228 -> done
230 -> done
231 -> done
232 -> done
233 -> done
234 -> done


In [ ]:
all_annotations = pd.concat([
    pd.read_csv(os.path.join(output_path, f'{name}_annotation.csv'))
    for name in record_names
], ignore_index=True)

print(all_annotations['AAMI'].value_counts(dropna=False))
print(all_annotations.loc[all_annotations['AAMI'].isna(), 'Type'].value_counts())

AAMI
N      90631
Q       8043
V       7236
NaN     3153
S       2781
F        803
Name: count, dtype: int64
Type
+    1291
~     616
!     472
"     437
x     193
|     132
[       6
]       6
Name: count, dtype: int64
